# VitaGraph Research Notebook

Cross-validation, model comparison, and sensitivity analysis on the
synthetic cohort — useful as a template for how you might evaluate a
*real* dataset once one is plugged in (see `docs/ROADMAP.md`).

> All data below is synthetic (seeded random data), used to illustrate
> methodology, not to make any claim about real biological aging.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from vitagraph import BioAgeEstimator, SyntheticCohortGenerator
from vitagraph.synthetic_data import FEATURE_COLUMNS, TARGET_COLUMN

pd.set_option("display.precision", 3)


## Model comparison across backends

In [2]:
cohort = SyntheticCohortGenerator(seed=42).generate(num_samples=2000)
X, y = cohort[FEATURE_COLUMNS], cohort[TARGET_COLUMN]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rows = []
for model_type in ["linear", "random_forest", "gradient_boosting"]:
    est = BioAgeEstimator(model_type=model_type)
    cv = est.cross_validate(X_train, y_train, cv=5)
    est.train(X_train, y_train)
    test_metrics = est.evaluate(X_test, y_test)
    rows.append({"model": model_type, **{f"cv_{k}": v for k, v in cv.items()}, **test_metrics.as_dict()})

pd.DataFrame(rows)

2026-07-18 13:22:47 | INFO     | vitagraph.bio_age_estimator | 5-fold CV — MAE: 1.59 ± 0.08  R²: 0.982 ± 0.001


2026-07-18 13:22:47 | INFO     | vitagraph.bio_age_estimator | Training linear on 1600 samples (6 features)


2026-07-18 13:22:47 | INFO     | vitagraph.bio_age_estimator | Evaluation — MAE: 1.52  RMSE: 1.92  R²: 0.984


2026-07-18 13:22:49 | INFO     | vitagraph.bio_age_estimator | 5-fold CV — MAE: 2.27 ± 0.12  R²: 0.964 ± 0.003


2026-07-18 13:22:49 | INFO     | vitagraph.bio_age_estimator | Training random_forest on 1600 samples (6 features)


2026-07-18 13:22:50 | INFO     | vitagraph.bio_age_estimator | Evaluation — MAE: 2.24  RMSE: 2.85  R²: 0.964


2026-07-18 13:22:52 | INFO     | vitagraph.bio_age_estimator | 5-fold CV — MAE: 1.80 ± 0.06  R²: 0.978 ± 0.001


2026-07-18 13:22:52 | INFO     | vitagraph.bio_age_estimator | Training gradient_boosting on 1600 samples (6 features)


2026-07-18 13:22:53 | INFO     | vitagraph.bio_age_estimator | Evaluation — MAE: 1.75  RMSE: 2.19  R²: 0.979


,model,cv_cv_folds,cv_mae_mean,cv_mae_std,cv_r2_mean,cv_r2_std,mae,rmse,r2
0,linear,5,1.590,0.079,0.982,9.000e-04,1.520,1.923,0.984
1,random_forest,5,2.271,0.117,0.964,3.200e-03,2.237,2.849,0.964
2,gradient_boosting,5,1.802,0.059,0.978,8.000e-04,1.746,2.191,0.979


## Sensitivity to training-set size (learning curve)

In [3]:
from sklearn.model_selection import learning_curve

est = BioAgeEstimator(model_type="gradient_boosting")
train_sizes, train_scores, test_scores = learning_curve(
    est.model, X[FEATURE_COLUMNS], y, cv=5,
    train_sizes=np.linspace(0.1, 1.0, 6),
    scoring="neg_mean_absolute_error",
)

pd.DataFrame({
    "train_size": train_sizes,
    "train_mae": -train_scores.mean(axis=1),
    "cv_mae": -test_scores.mean(axis=1),
})

,train_size,train_mae,cv_mae
0,160,0.264,2.330
1,448,0.736,1.999
2,736,0.934,1.894
3,1024,1.037,1.842
4,1312,1.110,1.784
5,1600,1.177,1.762


## Seed sensitivity

Since every generator is seeded, re-running with different seeds shows
how much of the reported performance is an artifact of a single
synthetic draw versus a stable property of the (deterministic) label
function in `vitagraph.config.BioAgeFormulaWeights`.

In [4]:
results = []
for seed in range(5):
    c = SyntheticCohortGenerator(seed=seed).generate(num_samples=1000)
    Xs, ys = c[FEATURE_COLUMNS], c[TARGET_COLUMN]
    Xtr, Xte, ytr, yte = train_test_split(Xs, ys, test_size=0.2, random_state=seed)
    est = BioAgeEstimator(model_type="gradient_boosting")
    est.train(Xtr, ytr)
    m = est.evaluate(Xte, yte)
    results.append({"seed": seed, **m.as_dict()})

pd.DataFrame(results)

2026-07-18 13:23:02 | INFO     | vitagraph.bio_age_estimator | Training gradient_boosting on 800 samples (6 features)


2026-07-18 13:23:02 | INFO     | vitagraph.bio_age_estimator | Evaluation — MAE: 1.79  RMSE: 2.32  R²: 0.975


2026-07-18 13:23:02 | INFO     | vitagraph.bio_age_estimator | Training gradient_boosting on 800 samples (6 features)


2026-07-18 13:23:02 | INFO     | vitagraph.bio_age_estimator | Evaluation — MAE: 1.74  RMSE: 2.19  R²: 0.978


2026-07-18 13:23:02 | INFO     | vitagraph.bio_age_estimator | Training gradient_boosting on 800 samples (6 features)


...

2026-07-18 13:23:03 | INFO     | vitagraph.bio_age_estimator | Training gradient_boosting on 800 samples (6 features)


2026-07-18 13:23:03 | INFO     | vitagraph.bio_age_estimator | Evaluation — MAE: 1.92  RMSE: 2.41  R²: 0.974


2026-07-18 13:23:03 | INFO     | vitagraph.bio_age_estimator | Training gradient_boosting on 800 samples (6 features)


2026-07-18 13:23:03 | INFO     | vitagraph.bio_age_estimator | Evaluation — MAE: 1.84  RMSE: 2.33  R²: 0.974


,seed,mae,rmse,r2
0,0,1.793,2.324,0.975
1,1,1.742,2.188,0.978
2,2,1.703,2.138,0.979
3,3,1.919,2.407,0.974
4,4,1.837,2.330,0.974


## Scope note

This notebook evaluates a *synthetic, deterministic* label. Held-out R²
values above are expected to be high because the label is generated
from a documented linear-ish formula over the same features used to
predict it (see `vitagraph/config.py::BioAgeFormulaWeights`). These
numbers demonstrate correct model-fitting mechanics on known
ground truth — they say nothing about performance on a real
biological-age task, which is a materially harder, noisier problem.
See `docs/METHODOLOGY.md` for the full scope statement and
`docs/ROADMAP.md` for the plan to plug in real (properly licensed,
IRB-approved) datasets.